# Does the phase coherence depend on azimuth? Synthetic test

## The reviewer's question
The phase-coherence method gives **one** fall-off frequency ff per earthquake for the whole array. Amplitude-based methods give one corner frequency **per station**, so one can plot those against
azimuth and see whether they scatter randomly or follow a pattern. With a single number for the array, the reviewer cannot tell whether the fall-off is a real source signal or just random
fluctuation in the spectra. The request: show the coherence of **station pairs** against the azimuth difference of the pair, on synthetics (does the method behave as expected?) and on real data.

## The physical idea
* **Point source:** every station sees the same pulse, so all stations have the same phase and the coherence of every pair is 1 at all frequencies.
* **Finite rupture:** the apparent source time function of a station depends on the direction of its ray *projected on the fault plane*. Stations looking at the rupture from similar directions see similar pulses
  and stay coherent to higher frequency. Stations looking from very different directions see different pulses and lose coherence at lower frequency.
* **Noise** does not care about azimuth. An ordered dependence on azimuth is therefore the fingerprint of a real source signal, which is what the reviewer asks for.

## What the code computes
1. After `calcxc`, `calcmvout`, `cpfromr`, every station has a phase `phi_i(f)`: the moveout-corrected phase of the cross-spectrum (target relative to reference).
2. **Pair coherence** `c_ij(f) = cos(phi_i - phi_j)`: 1 for the same phase, 0 for phases at right angles, -1 for opposite phases.
3. The mean of `c` over **all** pairs is the whole-array coherence you already use (the black curve; `identity check` printed below must be about 1e-15), so the pair test is a decomposition of the number behind ff.
4. The pairs are grouped by **|Δaz|**, the smallest angle between the two station azimuths (0-180 deg), and averaged in each group. The shaded bands are a leave-one-station-out jackknife.

## What to expect
* below about 0.5 ff: every group is near 1, nothing separates the pairs;
* around and above ff: the groups fan out in order: close pairs stay coherent, widely separated pairs collapse first (the whole array falls where those have already gone);
* far above ff: values near 0, and negative where phases flip (in the ideal case without noise). The widest pairs can come back up: |Δaz| is only a proxy for the real separation of two stations *in the fault plane* (|Δp|, a later refinement).

## The two figures
1. **Coherence against frequency**, one curve per |Δaz| group, the whole array in black, ff dashed.
2. **Heatmap**: coherence as a function of frequency and |Δaz| (blue = coherent, grey = 0, red = negative; the black line marks coherence = 0.5).

## The synthetic data
The synthetic data come from your own pipeline (`testappstf` then `phscoh`), as in `stress_drop_demo.ipynb`, with a dense array (`N_STATIONS`) and several random slip realisations.
They are saved **once** in a folder (`synthetic_data/`, one `.npz` file per realisation, plus a `synthetic_data.zip` of it) and can be read back later without `testappstf` or `phscoh`, or uploaded.
In section 1 choose ONE of the two options by putting a `#` in front of the other line.

In [ ]:
import os, importlib
import numpy as np
import matplotlib.pyplot as plt
import azimuth_coherence_tools as act
importlib.reload(act)                                  # picks up a replaced azimuth_coherence_tools.py without restarting the kernel
if not hasattr(act, 'save_synthetic'):
    raise ImportError(f'{act.__file__} is an OLD azimuth_coherence_tools.py (no save_synthetic). Copy the new file next to this notebook, '
                      'replacing that one, then restart the kernel and run again.')

plt.rcParams.update({'font.size': 11, 'figure.dpi': 110, 'savefig.dpi': 300})

# ---------------- where things are saved ----------------
DATA_DIR = 'synthetic_data'                 # the synthetic data (one .npz per realisation) are written to / read from this folder
FIG_DIR  = 'figures'                        # the figures are saved here

# ---------------- OPTION A only: how the synthetic data are generated ----------------
N_STATIONS = 37                             # strikes 0 ... 180 deg every 5 deg (the demo's N = 5 gives 0, 45, ..., 180)
RADII      = [100.0]                        # target radius (m)
CASES      = [('het', 5)]                   # (earthquake type, number of random realisations); add ('bwm', 1) for the other type of the demo
NSAMP      = 16                             # decimation factor of the demo
XC_KW      = dict(trange=[-0.5, 2.5], mk1='t0', mk2='t0', tpr='multi')     # as in the demo ('cosine' if 'multi' gives odd results)

# ---------------- analysis and figures (both options) ----------------
DAZ_EDGES = [0, 10, 30, 60, 90, 180]        # groups of |azimuth difference| (deg) of the coherence curves (figure 1)
HEAT_STEP = 5.0                             # bin width (deg) of the heatmap (figure 2); 5 deg = the station spacing here
SHOW      = 0                               # which realisation to plot: its position in the list (0 = first), or None for all

## 1. The synthetic data: choose ONE option

* **Option A, generate:** runs `testappstf` and `phscoh` (needs `phscoh.py`, `testappstf.py`, `syntheq.py` on the path), saves every realisation in `DATA_DIR`, then reads it back from there, so that A and B give identical results.
* **Option B, read:** reads the files already in `DATA_DIR`. Needs only `azimuth_coherence_tools.py`.

Each file holds the cross-spectra of the `xc` object (`xc`, `xcdata`, ...), `R`, `Cp`, the fall-off frequency `ffbest`, the station azimuths and take-off angles **in the order of the `xc` columns**,
the in-plane ray components, and the settings of the run.

In [ ]:
def _num(x):
    """first element as a float (a length-1 array cannot be converted with float() in recent NumPy)"""
    return float(np.asarray(x, float).reshape(-1)[0])


def one_realisation(phscoh, testappstf, eqtyp, radius, seed):
    np.random.seed(seed)
    seqs, seqs2 = testappstf.appstfcalc(N=N_STATIONS, eqtyp=eqtyp, rad=np.array([radius]))
    seqs, seqs2 = seqs[0], seqs2[0]
    st1, st2 = seqs.obswaveforms(), seqs2.obswaveforms()
    st1.decimate(factor=NSAMP)
    st2.decimate(factor=NSAMP)
    xc = phscoh.calcxc(st1, st2, **XC_KW)
    xc.calcmvout()
    xc.cpfromr()
    ff_source = act.safe_pickffreq(xc)                                      # = xc.pickffreq(), with a fallback if phscoh.pickffreq fails under your NumPy
    az = act.as_vector(seqs.ststrike)                                       # flat array whatever shape ststrike has
    tk = act.as_vector(seqs.sttakeang)
    ordered = act.check_station_order(phscoh, st1, st2, verbose=(seed == 0), **XC_KW)     # are the columns of xc in the order of the stations (az)?
    if not ordered:
        order = act.resolve_station_order(phscoh, st1, st2, **XC_KW)        # if not (phscoh sorts the stations itself): find the order, one station at a time
        az, tk = az[order], (tk[order] if len(tk) > 1 else tk)
    P = act.inplane_coords(az, tk, seqs.strike, seqs.dip)                   # ststrike is taken as the azimuth; with strike 0 and dip 90 (the demo) the convention does not matter
    assert np.allclose(act.as_vector(seqs.ststrike), act.as_vector(seqs2.ststrike)), 'target and reference have different station geometry'
    meta = dict(eqtyp=eqtyp, radius=radius, seed=seed, n_stations=len(az), nsamp=NSAMP, xc_kw=XC_KW, ff_source=ff_source, order_resolved=(not ordered),
                strike=_num(seqs.strike), dip=_num(seqs.dip), target_axes=[_num(seqs.a), _num(seqs.b)], reference_axes=[_num(seqs2.a), _num(seqs2.b)])
    return xc, az, tk, P, meta


def generate_synthetic():
    """OPTION A: run testappstf + phscoh for every case, save each realisation in DATA_DIR and read it back."""
    import phscoh, testappstf                                               # only needed here
    data, shown = {}, False
    for eqtyp, nreal in CASES:
        for radius in RADII:
            for seed in range(nreal):
                name = f'{eqtyp}_a{radius:.0f}m_seed{seed:02d}'
                try:
                    xc, az, tk, P, meta = one_realisation(phscoh, testappstf, eqtyp, radius, seed)
                except Exception as e:
                    import traceback
                    print('skipped', name, '-', type(e).__name__, e)
                    if not shown:                                           # first failure: show where it happened
                        traceback.print_exc()
                        shown = True
                    continue
                if not data:
                    print(f'{name}: {len(az)} stations, strikes {az[:4]} ..., take-off angles {np.unique(tk)}, target axes {meta["target_axes"][0]:.0f} x {meta["target_axes"][1]:.0f} m, '
                          f'reference axes {meta["reference_axes"][0]:.0f} x {meta["reference_axes"][1]:.0f} m, whole-array ff = {xc.ffbest:.1f} Hz')
                path = act.save_synthetic(os.path.join(DATA_DIR, name + '.npz'), xc, az, tk, P, dict(meta, name=name))
                data[name] = act.load_synthetic(path)                       # read back from the file: A and B then run exactly the same code
    print(len(data), 'realisations saved in', os.path.abspath(DATA_DIR))
    import shutil                                                           # one zip file of the folder: easy to share or upload
    parent, base = os.path.dirname(os.path.abspath(DATA_DIR)), os.path.basename(os.path.abspath(DATA_DIR))
    print('zipped for sharing / upload:', shutil.make_archive(os.path.join(parent, base), 'zip', root_dir=parent, base_dir=base))
    return data

In [ ]:
# ===== choose ONE: put a # in front of the line you do NOT want =====
data = generate_synthetic()                        # OPTION A: generate with testappstf + phscoh, save in DATA_DIR, read back
# data = act.load_synthetic_folder(DATA_DIR)       # OPTION B: read the files already saved in DATA_DIR (no testappstf / phscoh needed)

import pandas as pd
display(pd.DataFrame([dict(name=n, **{'stations': len(d['az']), 'tapers': (d['xc'].xc.shape[2] if d['xc'].xc.ndim == 3 else 1), 'ff (Hz)': round(d['xc'].ffbest, 2),
                                      'ff from': d['meta'].get('ff_source'), 'file': os.path.basename(d['path'])}) for n, d in data.items()]).set_index('name'))

## 2. Pair coherence for every realisation

`analyse_array` takes the station phasors from `xc`, forms `cos(phi_i - phi_j)` for all pairs and averages the pairs of each |Δaz| group. It checks which taper combination reproduces `xc.R` and prints it,
together with the identity check (first realisation only). No permutation test here (`n_perm=0`); it comes back with the band figures.

In [ ]:
results = {}
for k, (name, d) in enumerate(data.items()):
    results[name] = act.analyse_array(d['xc'], d['az'], P=d['P'], daz_edges=DAZ_EDGES, n_perm=0, verbose=(k == 0))

def title_of(name):
    m = data[name]['meta']
    return f"{m.get('eqtyp', '')}, a = {m.get('radius', float('nan')):.0f} m, realisation {m.get('seed', '?')}   ({len(data[name]['az'])} stations, ff = {results[name]['ff']:.1f} Hz)"

names = list(results)
show = names if SHOW is None else [names[SHOW]]
os.makedirs(FIG_DIR, exist_ok=True)

## Figure 1: coherence against frequency

Each coloured curve is the mean coherence of all station pairs whose azimuths differ by the amount in the legend (light = close, dark = far apart); the shaded band is ±1 standard error (SE) from a **leave-one-station-out jackknife**.
Black is the mean over all pairs, the whole-array coherence behind ff (dashed). Expect: all curves at 1 at low frequency, then the dark ones leave 1 first.

**About the shaded band.** It is a *jackknife*, not a bootstrap: it is deterministic (no random draws) and uses delete-one resampling of the **stations**. For a group of pairs, the mean coherence is recomputed
with all pairs that contain station `s` removed, once for each station `s`, giving `k` values `m_(s)`. Then `SE = sqrt( (k-1)/k * sum_s (m_(s) - mean(m))^2 )`. Stations are resampled, not pairs, because pairs that share a station are not independent:
resampling pairs would make the band too narrow. The band is ±1 SE (roughly a 68% interval if the error is close to normal), not a 95% interval.

In [ ]:
for name in show:
    fig = act.plot_coherence(results[name], 'daz', title_of(name))
    fig.savefig(os.path.join(FIG_DIR, f'fig1_coherence_{name}.png'), bbox_inches='tight')
    plt.show()

## Figure 2: coherence as a function of frequency and azimuth difference

The same numbers as figure 1, but with every |Δaz| bin as its own row. Reading from left to right at a fixed |Δaz|: coherent (blue) at low frequency, then fading through grey to red. The black line (coherence = 0.5) should
bend to **lower** frequency as |Δaz| grows. Rows near 180 deg hold very few pairs (180 deg is a single pair), so they are noisy.

In [ ]:
for name in show:
    fig = act.plot_heatmap_figure(results[name], 'daz', step=HEAT_STEP, start=HEAT_STEP / 2, title=title_of(name))
    fig.savefig(os.path.join(FIG_DIR, f'fig2_heatmap_{name}.png'), bbox_inches='tight')
    plt.show()